In [ ]:
import osmnx as ox
#yol ağı verisinin indirilmesi
# Bursa polygonunu al 
place = "Bursa, Turkey"
G = ox.graph_from_place(place, network_type="drive")

# Kenar tablosunu DataFrame'e çevir
edges = ox.graph_to_gdfs(G, nodes=False, edges=True)

# Hangi sütunlar var bakalım
print(edges.columns)

import pandas as pd

def parse_speed(val):
    if isinstance(val, list):  
        val = val[0]
    if val is None:
        return None
    try:
        return int(str(val).split()[0])  #
    except:
        return None

edges["speed_kph"] = edges["maxspeed"].apply(parse_speed)

# highway sütunundaki listeleri stringe çevir
edges["highway"] = edges["highway"].apply(
    lambda x: x[0] if isinstance(x, list) else x
)


speed_defaults = {
    "motorway": 100,
    "trunk": 90,
    "primary": 70,
    "secondary": 50,
    "tertiary": 40,
    "residential": 30,
    "service": 20
}

edges["speed_kph"] = edges.apply(
    lambda r: speed_defaults.get(r["highway"], 40) if pd.isna(r["speed_kph"]) else r["speed_kph"],
    axis=1
)


# dakika cinsinden süre
edges["minutes"] = edges["length"] / (edges["speed_kph"] * 1000 / 60)

print(edges[["highway", "length", "speed_kph", "minutes"]].head())

#dosya büyük olduğu için başka yere 
edges.to_file(
    r"C:\Users\Monster\Desktop\case_veriler\bursa_roads_with_speed.shp",
    driver="ESRI Shapefile"
)


Index(['osmid', 'highway', 'oneway', 'reversed', 'length', 'geometry', 'lanes',
       'name', 'ref', 'bridge', 'tunnel', 'junction', 'maxspeed', 'access',
       'width'],
      dtype='object')
                               highway       length  speed_kph    minutes
u         v           key                                                
382957847 10143371658 0       tertiary    69.761706       40.0   0.104643
          382957801   0       tertiary   128.921672       40.0   0.193383
          10143371666 0    residential    99.384384       30.0   0.198769
382958032 2481951068  0       tertiary   254.434682       40.0   0.381652
          3147250661  0       tertiary  7529.623392       40.0  11.294435
